In [1]:
# UNIVERSAL GPU CONFIGURATION FOR TENSORFLOW
# ================================================================
!pip install tensorflow[and-cuda]



import tensorflow as tf
import os

# Configure TensorFlow to use GPU automatically everywhere
def configure_gpu():
    # Get available GPUs
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            # Enable memory growth to avoid allocating all memory at once
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            
            # Set logical GPU device configuration
            logical_gpus = tf.config.list_logical_devices('GPU')
            
            print(f"✅ GPU Configuration Successful!")
            print(f"📊 Found {len(gpus)} physical GPU(s), {len(logical_gpus)} logical GPU(s)")
            
            # Set GPU as default device for all operations
            tf.config.set_soft_device_placement(True)
            
            # Verify GPU is available and will be used
            print("🎯 TensorFlow will automatically use GPU for all operations")
            
        except RuntimeError as e:
            print(f"⚠️ GPU configuration error: {e}")
            print("🔧 Falling back to CPU")
    else:
        print("❌ No GPU found - Using CPU")
    
    return len(gpus) > 0

# Set environment variable to force GPU usage
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Call the configuration function
gpu_available = configure_gpu()

# Optional: Set GPU memory limit (adjust as needed)
if gpu_available:
    try:
        # Limit to 8GB if you have memory issues
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=8192)]
        )
        print("🔧 GPU memory limit set to 8GB")
    except:
        print("⚠️ Could not set memory limit - using default")

print("🚀 TensorFlow is ready! All operations will use GPU automatically.")

Defaulting to user installation because normal site-packages is not writeable


2025-11-14 15:25:46.962835: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ GPU Configuration Successful!
📊 Found 1 physical GPU(s), 1 logical GPU(s)
🎯 TensorFlow will automatically use GPU for all operations
⚠️ Could not set memory limit - using default
🚀 TensorFlow is ready! All operations will use GPU automatically.


I0000 00:00:1763114150.259908  152774 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9819 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [2]:
import pandas as pd
import numpy as np

folder = "/media/nasc/LocalDisk2/datasup"

# Load dataset sizes (RAM tiny)
# df_wel  = pd.read_csv(f"{folder}/welfake_clean.csv")
# df_net  = pd.read_csv(f"{folder}/fakenewsnet_clean.csv")
df_pred = pd.read_csv(f"{folder}/news_clean.csv")

# n_wel  = len(df_wel)
# n_net  = len(df_net)
n_pred = len(df_pred)

MAX_LEN = 300
EMB_DIM = 300

# ✅ Load memmap safely (zero RAM usage)
'''X_wel_sup = np.memmap(f"{folder}/welfake_sup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_wel, MAX_LEN, EMB_DIM))

X_net_sup = np.memmap(f"{folder}/fakenewsnet_sup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_net, MAX_LEN, EMB_DIM))'''

X_pred_sup = np.memmap(f"{folder}/fakepred_sup_seq.dat",
                         dtype="float32", mode="r",
                         shape=(n_pred, MAX_LEN, EMB_DIM))

# ✅ y labels are small → load normally
'''y_wel  = np.load(f"{folder}/welfake_labels.npy")
y_net  = np.load(f"{folder}/fakenewsnet_labels.npy")'''
y_pred = np.load(f"{folder}/fakepred_labels.npy")

print("✅ All datasets loaded via memmap")


✅ All datasets loaded via memmap


In [3]:



def make_dataset(X, y, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [4]:
from tensorflow.keras import layers, models, regularizers

MAX_LEN = 300         # shape is (N, 300, 300)
EMB_DIM = 300
L2_LAMBDA = 0.01
NUM_CLASSES = 2


def build_cnn_lstm():
    model = models.Sequential([
        layers.Input(shape=(MAX_LEN, EMB_DIM)),
        
        layers.Conv1D(64, 4, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.Conv1D(64, 3, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        
        layers.MaxPooling1D(pool_size=2),
        
        layers.LSTM(50, return_sequences=True,
                    kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.LSTM(30, kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        
        # Keep 2 outputs but use softmax
        layers.Dense(2, activation='softmax')  # Use softmax for multi-class
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',  # Use this for 1D integer labels
        metrics=['accuracy']
    )
    return model


In [5]:
def train_model(X, y, name, batch_size=32, epochs=10, save_dir="dataunsup"):
    import tensorflow as tf
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    import os

    print(f"\n🚀 Training {name} using GPU streaming...")

    # Ensure lengths match
    min_len = min(len(X), len(y))
    X, y = X[:min_len], y[:min_len]

    # Split only indices (not full arrays)
    idx_train, idx_test = train_test_split(
        np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
    )

    y_train, y_test = y[idx_train], y[idx_test]
    print(f"📊 Train size: {len(idx_train)} | Test size: {len(idx_test)}")

    # Generator to stream batches directly from memmap
    def data_gen(idxs, batch_size):
        n = len(idxs)
        while True:
            np.random.shuffle(idxs)
            for i in range(0, n, batch_size):
                batch_idx = idxs[i:i+batch_size]
                yield X[batch_idx], y[batch_idx]

    # TensorFlow Datasets (GPU-optimized streaming)
    ds_train = tf.data.Dataset.from_generator(
        lambda: data_gen(idx_train, batch_size),
        output_signature=(
            tf.TensorSpec(shape=(None, 300, 300), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int64)
        )
    ).prefetch(tf.data.AUTOTUNE)

    ds_test = tf.data.Dataset.from_generator(
        lambda: data_gen(idx_test, batch_size),
        output_signature=(
            tf.TensorSpec(shape=(None, 300, 300), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int64)
        )
    ).prefetch(tf.data.AUTOTUNE)

    # Build CNN-LSTM model
    model = build_cnn_lstm()

    # Train safely on GPU
    history = model.fit(
        ds_train,
        validation_data=ds_test,
        steps_per_epoch=len(idx_train)//batch_size,
        validation_steps=len(idx_test)//batch_size,
        epochs=epochs,
        verbose=1
    )

    # Evaluate model
    preds = model.predict(ds_test, steps=len(idx_test)//batch_size)
    y_pred = np.argmax(preds, axis=1)

    acc = accuracy_score(y_test[:len(y_pred)], y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test[:len(y_pred)], y_pred, average='binary'
    )

    print(f"\n📊 Results for {name}:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    # Save model
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f"cnn_lstm_{name}.h5")
    model.save(model_path)

    print(f"✅ Model saved at: {model_path}")

    # Save metrics for later comparison
    metrics_path = os.path.join(save_dir, f"{name}_metrics.txt")
    with open(metrics_path, "w") as f:
        f.write(f"Accuracy : {acc:.4f}\n")
        f.write(f"Precision: {prec:.4f}\n")
        f.write(f"Recall   : {rec:.4f}\n")
        f.write(f"F1 Score : {f1:.4f}\n")

    print(f"📁 Metrics saved at: {metrics_path}")

    return history, model_path


In [12]:
train_model(X_wel_sup,   y_wel, "welfake_sup")


🚀 Training welfake_sup...
Epoch 1/10


2025-11-10 13:50:40.897861: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91500


   1804/Unknown 43s 22ms/step - accuracy: 0.7263 - loss: 0.9308

2025-11-10 13:51:22.285252: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-11-10 13:51:22.285536: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/Const_6/_10]]
2025-11-10 13:51:22.285543: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16574567108115544188
2025-11-10 13:51:22.285546: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2849902868833524261
2025-11-10 13:51:22.285550: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 18076603602990399498
2025-11-10 13:51:22.285554: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 885156473793

1804/1804 ━━━━━━━━━━━━━━━━━━━━ 51s 26ms/step - accuracy: 0.7929 - loss: 0.5508 - val_accuracy: 0.9538 - val_loss: 0.1882
Epoch 2/10
   4/1804 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - accuracy: 0.9792 - loss: 0.1468  

...

1802/1804 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9659 - loss: 0.1131

2025-11-10 13:58:29.355592: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2849902868833524261
2025-11-10 13:58:29.356238: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16574567108115544188
2025-11-10 13:58:29.356421: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 18076603602990399498
2025-11-10 13:58:29.356429: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8851564737937188218


1804/1804 ━━━━━━━━━━━━━━━━━━━━ 47s 26ms/step - accuracy: 0.9650 - loss: 0.1142 - val_accuracy: 0.9659 - val_loss: 0.1146


2025-11-10 13:58:37.163200: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16532352252522359325
2025-11-10 13:58:37.163846: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14704503627546248959
2025-11-10 13:58:37.163859: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8830374960976881056


✅ Saved model cnn_lstm_welfake_sup.h5


In [13]:
train_model(X_net_sup,   y_net, "fakenewsnet_sup")


🚀 Training fakenewsnet_sup...
Epoch 1/10
    580/Unknown 15s 23ms/step - accuracy: 0.7449 - loss: 1.2938

2025-11-10 13:59:59.364140: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2849902868833524261
2025-11-10 13:59:59.364603: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16574567108115544188
2025-11-10 13:59:59.364614: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8851564737937188218
/home/nasc/ak/env/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


580/580 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.7511 - loss: 0.7853 - val_accuracy: 0.7519 - val_loss: 0.5605
Epoch 2/10
  4/580 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.7389 - loss: 0.5752  

2025-11-10 14:00:02.153613: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16532352252522359325
2025-11-10 14:00:02.154394: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14704503627546248959
2025-11-10 14:00:02.154410: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8830374960976881056


579/580 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7501 - loss: 0.5638

...

580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7501 - loss: 0.5629

2025-11-10 14:02:24.864714: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16574567108115544188
2025-11-10 14:02:24.865381: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2849902868833524261
2025-11-10 14:02:24.865565: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 18076603602990399498
2025-11-10 14:02:24.865582: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8851564737937188218


580/580 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.7519 - loss: 0.5608 - val_accuracy: 0.7519 - val_loss: 0.5607


2025-11-10 14:02:27.368518: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16532352252522359325
2025-11-10 14:02:27.369354: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14704503627546248959
2025-11-10 14:02:27.369367: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8830374960976881056


✅ Saved model cnn_lstm_fakenewsnet_sup.h5


In [6]:
train_model(X_pred_sup,   y_pred, "newspred_sup")


🚀 Training newspred_sup using GPU streaming...
📊 Train size: 5068 | Test size: 1267
Epoch 1/10


2025-11-14 15:26:30.426457: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91500


158/158 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.8973 - loss: 1.1616 - val_accuracy: 0.9022 - val_loss: 0.3967
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9015 - loss: 0.3448 - val_accuracy: 0.9014 - val_loss: 0.3256
Epoch 3/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9015 - loss: 0.3243 - val_accuracy: 0.9022 - val_loss: 0.3209
Epoch 4/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9007 - loss: 0.3258 - val_accuracy: 0.8998 - val_loss: 0.3260
Epoch 5/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9007 - loss: 0.3238 - val_accuracy: 0.9006 - val_loss: 0.3238
Epoch 6/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9035 - loss: 0.3189 - val_accuracy: 0.9006 - val_loss: 0.3237
Epoch 7/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9023 - loss: 0.3217 - val_accuracy: 0.9014 - val_loss: 0.3229
Epoch 8/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9003 - loss: 0.3249 - val_accuracy: 0.90

/home/nasc/ak/env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



📊 Results for newspred_sup:
Accuracy : 0.9006
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000
✅ Model saved at: dataunsup/cnn_lstm_newspred_sup.h5
📁 Metrics saved at: dataunsup/newspred_sup_metrics.txt


(<keras.src.callbacks.history.History at 0x7f39181a4a00>,
 'dataunsup/cnn_lstm_newspred_sup.h5')

In [8]:
# Quick test to verify your model works
import numpy as np
from tensorflow.keras.models import load_model

# Load the model
model_path = f"{folder}/cnn_lstm_welfake_sup.h5"
model = load_model(model_path)

print("✅ Model loaded successfully!")
print("Model summary:")
model.summary()

# Test with a single sample
print("\n🧪 Testing with random sample...")
test_sample = np.random.rand(1, 300, 300).astype(np.float32)  # Shape: (1, 300, 300)
prediction = model.predict(test_sample)

print(f"Raw prediction: {prediction}")
print(f"Prediction shape: {prediction.shape}")
print(f"Probability: {prediction[0][0]:.4f}")
print(f"Predicted class: {'Fake' if prediction[0][0] > 0.5 else 'Real'}")

✅ Model loaded successfully!
Model summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 297, 64)        │        76,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 295, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 147, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 147, 50)        │        23,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 30)             │         9,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            62 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,000 (476.57 KB)

 Trainable params: 121,998 (476.55 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


🧪 Testing with random sample...


2025-11-10 14:10:24.083786: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91500


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 593ms/step
Raw prediction: [[0.00837536 0.99162465]]
Prediction shape: (1, 2)
Probability: 0.0084
Predicted class: Real


In [9]:
import numpy as np
from tensorflow.keras.models import load_model

# Load your saved model
model = load_model(f"{folder}/cnn_lstm_welfake_sup.h5")

def test_single_sample(model, sample_input):
    """
    Test the model with a single sample
    """
    # Ensure the input has the right shape (1, 300, 300)
    if len(sample_input.shape) == 2:
        sample_input = np.expand_dims(sample_input, axis=0)
    
    prediction = model.predict(sample_input)
    
    # For binary classification with sigmoid output
    probability = prediction[0][0]  # Since output is (1, 1)
    predicted_class = 1 if probability > 0.5 else 0
    
    print(f"Prediction probability: {probability:.4f}")
    print(f"Predicted class: {predicted_class} ({'Fake' if predicted_class == 1 else 'Real'})")
    
    return predicted_class, probability

# Test with a random sample from your test set
def test_with_test_data(model, X_test, y_test, num_samples=5):
    """
    Test the model with samples from test data
    """
    print("🧪 Testing with test data samples...")
    
    for i in range(min(num_samples, len(X_test))):
        print(f"\n--- Sample {i+1} ---")
        sample = X_test[i]
        true_label = y_test[i]
        
        pred_class, probability = test_single_sample(model, sample)
        
        print(f"True label: {true_label} ({'Fake' if true_label == 1 else 'Real'})")
        print(f"Correct: {pred_class == true_label}")

# Usage
# test_with_test_data(model, X_test, y_test, num_samples=5)

In [10]:
def real_time_testing(model):
    """
    Interactive testing function
    """
    print("🎯 Real-time Testing Mode")
    print("Enter 'quit' to exit")
    
    while True:
        user_input = input("\nPress Enter to test with random sample or 'quit' to exit: ")
        
        if user_input.lower() == 'quit':
            break
            
        # Generate random sample (replace with your actual data loading)
        sample = np.random.randn(300, 300).astype(np.float32)
        
        print("Testing sample...")
        pred_class, probability = test_single_sample(model, sample)
        
        print(f"Result: {pred_class} ({'Fake' if pred_class == 1 else 'Real'}) with confidence {probability:.4f}")

# Start interactive testing
real_time_testing(model)

🎯 Real-time Testing Mode
Enter 'quit' to exit



Press Enter to test with random sample or 'quit' to exit:  akshay is gay because he was seen with a guy in hotel last night.


Testing sample...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
Prediction probability: 0.1743
Predicted class: 0 (Real)
Result: 0 (Real) with confidence 0.1743



Press Enter to test with random sample or 'quit' to exit:  Elon musk died


Testing sample...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Prediction probability: 0.3326
Predicted class: 0 (Real)
Result: 0 (Real) with confidence 0.3326



Press Enter to test with random sample or 'quit' to exit:  akshay is gay


Testing sample...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Prediction probability: 0.9530
Predicted class: 1 (Fake)
Result: 1 (Fake) with confidence 0.9530



Press Enter to test with random sample or 'quit' to exit:  akshay is gay because he was seen with a girl in hotel last night.


Testing sample...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction probability: 0.0250
Predicted class: 0 (Real)
Result: 0 (Real) with confidence 0.0250



Press Enter to test with random sample or 'quit' to exit:  quit
